### Profiling



- Profiling shows why a model is slow by identifying bottlenecks like slow data loading, heavy GPU ops, or memory issues.  
- It measures time, memory, and hardware utilization for each part of training or inference.  
- Profiling gives deep diagnostics (operator-level insights), while benchmarking measures overall performance like latency and throughput.  
- Time analysis splits CPU time vs GPU time to detect whether workloads are CPU-bound or GPU-bound.  
- Memory profiling tracks RAM/VRAM usage to find leaks, fragmentation, and optimize batch size or data transfer.  
- Compute utilization checks how efficiently hardware is used using metrics like FLOPS utilization and kernel occupancy.  
- Low GPU occupancy means hardware is underutilized and performance can still be improved.  
- Profiling enables data-driven optimization, reducing cloud cost and improving application responsiveness.

-  FLOPS Utilization: Measures how much of the hardware’s maximum compute power is actually being used. Higher utilization = better efficiency.
Formula: Compute Utilization = Model’s Achieved FLOPS / Hardware’s Peak FLOPS
-  Kernel Occupancy: Measures the percentage of GPU threads actively running at a time. Higher occupancy = less idle GPU resources.
Formula: Occupancy = Active Threads / Maximum Threads

Profiling Fundamentals 
- By transforming raw profiling logs into interactive timelines and flame graphs, you can make data-driven decisions to systematically improve model performance.

* Profiling identifies performance bottlenecks in training and inference.
* Training focuses on high throughput, while inference focuses on low latency.
* A common issue is data loading bottlenecks, where GPUs wait idle for data from disk or CPU preprocessing.
* Fixes include more dataloader workers, pinned memory, and pipelined preprocessing.
* Profilers detect operator hotspots like `conv2d`, `matmul`, and activations consuming most compute time.
* Flame graphs visualize execution time; wider bars indicate major hotspots.
* In multi-GPU training (DDP), gradient synchronization (`all-reduce`) creates communication overhead.
* GPU execution is asynchronous — CPU launches kernels without waiting for completion.
* Use `torch.cuda.synchronize()` for accurate GPU timing measurements.
* Profiling helps focus optimization on the biggest bottlenecks for maximum speedup.


Tools and Frameworks for Profiling


* Performance optimization starts with collecting profiling data instead of guessing bottlenecks.
* Workflow: Instrument → Collect traces → Visualize → Optimize → Repeat.
* `torch.profiler.profile()` records CPU ops, GPU kernels, and memory activity.

```python
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    profile_memory=True,
    with_stack=True,
    with_shapes=True
) as prof:
    train_step()
```

* `profile_memory=True` tracks memory allocation and peak usage.
* `with_stack=True` captures Python call stacks.
* `with_shapes=True` records tensor dimensions.
* `prof.key_averages().table()` shows top time-consuming operators.

```python
print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=10
))
```

* Profiling data can be visualized using:

  * Chrome Trace (`trace.json`)
  * TensorBoard Profile tab

* Timeline View shows operation start/end times and idle gaps.

* White gaps in timeline often indicate waiting or data-loading bottlenecks.

* Flame Graph shows hierarchical function calls and self-time.

* Wider bars in flame graphs represent major performance hotspots.

* Optimization is iterative: profile → analyze → optimize → re-profile.

Free response:

* Timeline view is useful when diagnosing delays between operations, idle GPU time, or synchronization issues.
* Flame graphs are better for identifying which specific functions consume the most execution time.


* Profiling finds where code spends most time and resources.
* Optimization cycle: Profile → Analyze → Optimize → Re-profile.
* Run warm-up iterations before profiling for accurate results.
* `torch.profiler.profile()` records CPU/GPU operations.
* `prof.key_averages().table()` shows top costly operators.
```python
with torch.profiler.profile(...):
    train_step()

prof.key_averages().table(
    sort_by="self_cpu_time_total"
)
```
* Common hotspots: `conv2d`, `matmul`, `batch_norm`.
* Timeline View shows execution flow and idle gaps.
* Operator Breakdown shows costly ops and tensor shapes.
* Optimize major bottlenecks using fusion or faster algorithms.
* Re-profile to verify performance improvement.


# Demo 1 - Resnet-50 Training Profiling

| Phase     | Self CPU Time | Self CUDA Time | Main Bottlenecks                                               |
| --------- | ------------- | -------------- | -------------------------------------------------------------- |
| Training  | 735.986 ms    | 537.796 ms     | DataLoader, `aten::to`, convolution, batch norm, backward pass |
| Inference | 307.277 ms    | 337.072 ms     | DataLoader, convolution, batch norm                            |

### Top Operations Comparison

| Operation           | Training   | Inference  | Observation                         |
| ------------------- | ---------- | ---------- | ----------------------------------- |
| DataLoader          | 25.95% CPU | 41.14% CPU | Input pipeline overhead significant |
| `aten::to`          | 18.98% CPU | 9.18% CPU  | CPU→GPU transfer overhead           |
| `cudnn_convolution` | 47.5% CUDA | 75.5% CUDA | Main GPU compute hotspot            |
| Batch Normalization | 18.6% CUDA | 8.7% CUDA  | Higher during training              |
| Backward Pass       | Present    | Absent     | Major extra training cost           |

### Training vs Inference

| Metric          | Training | Inference                  |
| --------------- | -------- | -------------------------- |
| Forward Pass    | ✅        | ✅                          |
| Backward Pass   | ✅        | ❌                          |
| Optimizer Step  | ✅        | ❌                          |
| Memory Usage    | Higher   | Lower                      |
| Latency         | Higher   | Lower                      |
| GPU Utilization | Mixed    | Mostly convolution kernels |



The PyTorch Profiler and TensorBoard provide a detailed, visual breakdown of model performance, enabling developers to identify and optimize time-consuming operations on both the CPU and GPU.

In [ ]:
import torch.profiler

#training
CAPTURE_SCHEDULE = torch.profiler.schedule(wait=1, warmup=1, active=7, repeat=1)
def train_log(profiler_logs_dir):
    with torch.profiler.profile(
        activities=activities,
        schedule=CAPTURE_SCHEDULE,
        on_trace_ready=torch.profiler.tensorboard_trace_handler(profiler_logs_dir),
        record_shapes=True,
        profile_memory=True,
        with_stack=False, # optional: samples C++ operator stack
    ) as prof:
        for step, (x, y) in enumerate(train_loader):
            if step >= (1 + 1 + 7) * 1: # wait, warmup, active, repeat
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            prof.step() # tell profiler a step is finished

train_log(f"{LOGDIR}/train")

# inference
def infer_log(logdir):
    with torch.profiler.profile(
        activities=activities,
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=7, repeat=1),
        on_trace_ready=torch.profiler.tensorboard_trace_handler(logdir),
        record_shapes=True,
        profile_memory=True,
        with_stack=False, # optional: samples C++ operator stack
    ) as prof:
        # run a handful of micro-batches
        it = iter(test_loader)
        for step in range(12):
            x, _ = next(it)
            x = x.to(device, non_blocking=True)
            model(x)
            prof.step()

infer_log(f"{LOGDIR}/infer")

To find performance bottlenecks, we use the PyTorch Profiler. We configure it with a schedule (wait, warmup, active) to get stable measurements and define a _trace_handler to automatically save the results. The profiler wraps the training loop and records CPU and CUDA activity, which can be exported to TensorBoard to identify the most time-consuming operations.

Based on these experiments, we can conclude:

Top slow ops: Convolutions (aten::convolution), batch normalization, and activation functions are typically the slowest operations in CNNs.

Faster model: resnet18 often has higher throughput than resnet34 due to its fewer layers and parameters.

Batch size / workers: Larger batch sizes increase math per step, which improves throughput until limited by memory. More workers can help overlap CPU data loading with GPU computation, but diminishing returns are common.

Next step: To further improve performance, one could explore mixed-precision training with torch.cuda.amp.autocast, enable pin_memory=True in the DataLoader, and use a channels-last memory format for convolutions on CUDA.